In [0]:
orders_bronze = spark.table(
    "workspace.default.orders_bronze"
)

In [0]:
print("Rows:", orders_bronze.count())

In [0]:
orders_bronze.printSchema()

In [0]:
from pyspark.sql.functions import col, datediff

In [0]:
from pyspark.sql.functions import col, datediff, when

orders_silver = (
    orders_bronze
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),

        col("order_purchase_timestamp").alias("purchase_timestamp"),
        col("order_approved_at").alias("approved_timestamp"),
        col("order_delivered_carrier_date").alias("delivered_carrier_timestamp"),
        col("order_delivered_customer_date").alias("delivered_customer_timestamp"),
        col("order_estimated_delivery_date").alias("estimated_delivery_timestamp")
    )
    .withColumn(
        "delivery_days",
        datediff(
            col("delivered_customer_timestamp"),
            col("purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_delay_days",
        datediff(
            col("delivered_customer_timestamp"),
            col("estimated_delivery_timestamp")
        )
    )
    .withColumn(
        "date_sequence_valid",
        when(
            (
                col("approved_timestamp").isNotNull() &
                (col("approved_timestamp") < col("purchase_timestamp"))
            ) |
            (
                col("delivered_carrier_timestamp").isNotNull() &
                (col("delivered_carrier_timestamp") < col("purchase_timestamp"))
            ) |
            (
                col("delivered_customer_timestamp").isNotNull() &
                (col("delivered_customer_timestamp") < col("purchase_timestamp"))
            ),
            False
        ).otherwise(True)
    )
)

In [0]:
orders_silver.groupBy(
    "date_sequence_valid"
).count().show()

In [0]:
orders_silver.filter(
    col("date_sequence_valid") == False
).select(
    "order_id",
    "purchase_timestamp",
    "approved_timestamp",
    "delivered_carrier_timestamp",
    "delivered_customer_timestamp"
).show(10, truncate=False)

In [0]:
orders_silver.printSchema()

In [0]:
orders_silver.show(10, truncate=False)

In [0]:
orders_silver.select(
    "order_id",
    "order_status",
    "purchase_timestamp",
    "delivered_customer_timestamp",
    "estimated_delivery_timestamp",
    "delivery_days",
    "delivery_delay_days"
).filter(
    col("order_status") == "delivered"
).show(10, truncate=False)

In [0]:
orders_silver.filter(
    col("delivered_customer_timestamp").isNull()
).select(
    "order_id",
    "order_status",
    "purchase_timestamp",
    "delivered_customer_timestamp",
    "delivery_days",
    "delivery_delay_days"
).show(10, truncate=False)

In [0]:
print("Bronze rows:", orders_bronze.count())
print("Silver rows:", orders_silver.count())


In [0]:
duplicate_orders = (
    orders_silver
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate order IDs:", duplicate_orders.count())

In [0]:
(
    orders_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.orders_silver")
)

In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM workspace.default.orders_silver
""").show()

In [0]:
spark.sql("""
    SELECT *
    FROM workspace.default.orders_silver
    LIMIT 10
""").show(truncate=False)


In [0]:
customers_bronze = spark.table(
    "workspace.default.customers_bronze"
)
customers_silver = (
    customers_bronze
    .select(
        col("customer_id"),
        col("customer_unique_id"),
        col("customer_zip_code_prefix").alias("zip_code_prefix"),
        col("customer_city").alias("city"),
        col("customer_state").alias("state")
    )
)

In [0]:
customers_silver.show(10, truncate=False)

In [0]:
customers_silver.select(
    [
        col(c).isNull().cast("int").alias(c)
        for c in customers_silver.columns
    ]
).agg(
    *[
        __import__("pyspark.sql.functions", fromlist=["sum"]).sum(col(c)).alias(c)
        for c in customers_silver.columns
    ]
).show()

In [0]:
(
    customers_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.customers_silver"
    )
)

In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM workspace.default.customers_silver
""").show()

In [0]:
products_bronze = spark.table("workspace.default.products_bronze")
sellers_bronze = spark.table("workspace.default.sellers_bronze")
order_items_bronze = spark.table("workspace.default.order_items_bronze")
payments_bronze = spark.table("workspace.default.payments_bronze")
reviews_bronze = spark.table(
    "workspace.default.reviews_bronze"
)
geolocation_bronze = spark.table("workspace.default.geolocation_bronze")
category_translation_bronze = spark.table(
    "workspace.default.category_translation_bronze"
)

In [0]:
products_silver = (
    products_bronze
    .select(
        col("product_id"),
        col("product_category_name").alias("category_name"),
        col("product_name_lenght").alias("product_name_length"),
        col("product_description_lenght").alias("product_description_length"),
        col("product_photos_qty").alias("photos_qty"),
        col("product_weight_g").alias("weight_g"),
        col("product_length_cm").alias("length_cm"),
        col("product_height_cm").alias("height_cm"),
        col("product_width_cm").alias("width_cm")
    )
)



In [0]:
sellers_silver = (
    sellers_bronze
    .select(
        col("seller_id"),
        col("seller_zip_code_prefix").alias("zip_code_prefix"),
        col("seller_city").alias("city"),
        col("seller_state").alias("state")
    )
)

In [0]:
order_items_silver = (
    order_items_bronze
    .select(
        col("order_id"),
        col("order_item_id").alias("item_id"),
        col("product_id"),
        col("seller_id"),
        col("shipping_limit_date").alias("shipping_limit_timestamp"),
        col("price"),
        col("freight_value")
    )
)

In [0]:
payments_silver = (
    payments_bronze
    .select(
        col("order_id"),
        col("payment_sequential"),
        col("payment_type"),
        col("payment_installments"),
        col("payment_value").alias("payment_amount")
    )
    .withColumn(
        "payment_valid",
        when(
            (col("payment_amount") <= 0) |
            (col("payment_installments") < 1),
            False
        ).otherwise(True)
    )
)

In [0]:
payments_silver.filter(
    col("payment_valid") == False
).show(20, truncate=False)

In [0]:
from pyspark.sql.functions import col, to_timestamp

reviews_silver = (
    reviews_bronze
    .select(
        col("review_id"),
        col("order_id"),
        col("review_score").cast("int").alias("review_score"),
        col("review_comment_title").alias("comment_title"),
        col("review_comment_message").alias("comment_message"),
        to_timestamp(
            col("review_creation_date")
        ).alias("review_created_timestamp"),
        to_timestamp(
            col("review_answer_timestamp")
        ).alias("review_answered_timestamp")
    )
)

In [0]:
reviews_silver.printSchema()

In [0]:
from pyspark.sql.functions import col

reviews_bronze = spark.table(
    "workspace.default.reviews_bronze"
)

bad_creation = reviews_bronze.filter(
    ~col("review_creation_date").rlike(
        r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
    )
    & col("review_creation_date").isNotNull()
)

print("Bad creation dates:", bad_creation.count())

In [0]:
bad_creation.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)

In [0]:
bad_answer = reviews_bronze.filter(
    ~col("review_answer_timestamp").rlike(
        r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
    )
    & col("review_answer_timestamp").isNotNull()
)

print("Bad answer timestamps:", bad_answer.count())

In [0]:
bad_answer.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)

In [0]:
print("Bronze rows:", reviews_bronze.count())

In [0]:
(
    reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.reviews_silver"
    )
)

In [0]:
category_translation_silver = (
    category_translation_bronze
    .select(
        col("product_category_name"),
        col("product_category_name_english").alias(
            "category_name_english"
        )
    )
)

In [0]:
geolocation_silver = (
    geolocation_bronze
    .select(
        col("geolocation_zip_code_prefix").alias("zip_code_prefix"),
        col("geolocation_lat").alias("latitude"),
        col("geolocation_lng").alias("longitude"),
        col("geolocation_city").alias("city"),
        col("geolocation_state").alias("state")
    )
)

In [0]:
(
    products_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.products_silver")
)

(
    sellers_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.sellers_silver")
)

(
    order_items_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.order_items_silver")
)

(
    payments_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.payments_silver")
)

(
    reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.reviews_silver"
    )
)

(
    geolocation_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.geolocation_silver")
)

(
    category_translation_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.category_translation_silver"
    )
)

In [0]:
reviews_bronze.printSchema()

In [0]:
reviews_bronze.select(
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)

In [0]:
reviews_bronze.filter(
    ~col("review_score").cast("string").rlike("^[1-5]$")
).select(
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
).show(30, truncate=False)

In [0]:
silver_tables = [
    "orders_silver",
    "customers_silver",
    "products_silver",
    "sellers_silver",
    "order_items_silver",
    "payments_silver",
    "reviews_silver",
    "geolocation_silver",
    "category_translation_silver"
]

for table in silver_tables:
    count = spark.table(
        f"workspace.default.{table}"
    ).count()

    print(f"{table}: {count}")

In [0]:
from pyspark.sql.functions import col, to_timestamp

reviews_bronze = spark.table("workspace.default.reviews_bronze")

reviews_silver = (
    reviews_bronze
    .select(
        col("review_id"),
        col("order_id"),
        col("review_score").cast("int").alias("review_score"),
        col("review_comment_title").alias("comment_title"),
        col("review_comment_message").alias("comment_message"),
        to_timestamp(col("review_creation_date")).alias("review_created_timestamp"),
        to_timestamp(col("review_answer_timestamp")).alias("review_answered_timestamp")
    )
)

(
    reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.reviews_silver")
)

In [0]:
print(
    "Invalid review scores:",
    reviews_silver.filter(
        col("review_score").isNotNull() &
        ~col("review_score").isin(1, 2, 3, 4, 5)
    ).count()
)

In [0]:
print(
    "Total reviews:",
    reviews_silver.count()
)